# NLP Foundations: Tokenization, Word Embeddings & Attention

> **Focus Area:** ন্যাচারাল ল্যাঙ্গুয়েজ প্রসেসিং (Natural Language Processing - NLP Foundations)
> **Topics:** BPE টোকেনাইজেশন (Byte-Pair Encoding); ওয়ার্ড এমবেডিং (Word2Vec/GloVe); ট্রান্সফরমার অ্যাটেনশন (Self-Attention); সেন্টেন্স এমবেডিং ও কসাইন সিমিলারিটি
> **Achievement:** একটা **Semantic CV-to-Job Matcher**-এর বিল্ডিং ব্লকগুলো হাতে-কলমে দেখা — টেক্সট কীভাবে টোকেনে ভাঙে, টোকেন কীভাবে অর্থবহ ভেক্টরে রূপান্তর হয়, আর দুটো ভেক্টরের অর্থগত মিল কীভাবে মাপা হয়।

---

## 1. Topic: NLP Basics — Tokenization, Word Embeddings, Attention

কম্পিউটার সংখ্যা বোঝে, শব্দ বোঝে না। **NLP**-এর প্রথম কাজই হলো টেক্সটকে সংখ্যায় রূপান্তর করা, এমনভাবে যেন সেই সংখ্যাগুলো শব্দের **অর্থ** ধরে রাখে। আজকের নোটবুকে আমরা তিনটা ধাপ হাতে-কলমে দেখব, যেগুলো একে অপরের ওপর বিল্ড হয়েছে:

* **Tokenization (BPE)** — টেক্সটকে ছোট ছোট টুকরায় (token) ভাগ করা।
* **Word Embeddings (Word2Vec/GloVe)** — প্রতিটা শব্দকে একটা সংখ্যার ভেক্টরে রূপান্তর করা, যেখানে কাছাকাছি অর্থের শব্দ কাছাকাছি অবস্থানে থাকে।
* **Sentence Embeddings + Cosine Similarity** — পুরো বাক্য বা প্যারাগ্রাফকে একটা ভেক্টরে ধরে, দুটো টেক্সট কতটা অর্থগতভাবে কাছাকাছি তা মাপা।

এই তিনটা মিলেই তৈরি হয় আমাদের প্রজেক্ট **Semantic CV-to-Job Matcher**-এর ভিত্তি — যেটা কোনো জব ডেসক্রিপশনের সাথে একগুচ্ছ CV কতটা "মিলে যায়" তা বের করবে, শুধু কমন শব্দ গুনে নয়, বরং **অর্থ** বুঝে।

---

## 2. Why It Is Related

Week 2-এ আমরা শিখেছিলাম **Dot Product** এবং **Cosine Similarity** দিয়ে দুটো ভেক্টর কতটা "একই দিকে" নির্দেশ করে তা মাপা যায় (`week 2/Class 1 Lecture`-এর NumPy লেকচার মনে করুন)। NLP-তে সেই একই গণিত ব্যবহার হয় — শুধু ভেক্টরগুলো এখন সংখ্যার বদলে **শব্দ বা বাক্যের অর্থ** ধারণ করে।

* **Keyword Search** (traditional): "machine learning" শব্দটা CV-তে literally আছে কিনা চেক করে। "ML" বা "deep learning" লেখা থাকলে miss করে যাবে, যদিও অর্থ একই।
* **Semantic Search** (এই নোটবুকের টার্গেট): "machine learning", "ML", "deep learning" — এই তিনটাকেই কাছাকাছি ভেক্টর হিসেবে চেনে, কারণ ভেক্টরগুলো শব্দের **অর্থ** এনকোড করে, বানান নয়।

আজকের প্রায় সব বড় এআই সিস্টেম — ChatGPT, সার্চ ইঞ্জিন, রেকোমেন্ডেশন সিস্টেম — এই একই ভিত্তির ওপর দাঁড়িয়ে: **টেক্সটকে অর্থ-সংরক্ষণকারী ভেক্টরে রূপান্তর করা, তারপর ভেক্টরের মধ্যে সাদৃশ্য মাপা।**

---

## 3. How It Works

### 3.1 Tokenization — টেক্সটকে টুকরা করা

মডেল সরাসরি বাক্য পড়তে পারে না — আগে এটাকে ছোট ছোট **token**-এ ভাঙতে হয়। সবচেয়ে সহজ উপায় হলো শব্দ ধরে ধরে ভাঙা, কিন্তু এতে সমস্যা আছে: "running", "runner", "runs" — প্রতিটাকে আলাদা টোকেন ধরলে ভোকাবুলারি বিশাল হয়ে যায়, আবার নতুন শব্দ (out-of-vocabulary) এলে মডেল পুরোপুরি আটকে যায়।

**Byte-Pair Encoding (BPE)** এই সমস্যার সমাধান করে সাব-ওয়ার্ড (sub-word) লেভেলে ভেঙে:

```
"unhappiness" ──▶ ["un", "happi", "ness"]
"tokenization" ──▶ ["token", "ization"]
```

BPE কাজ করে সবচেয়ে বেশি রিপিট হওয়া ক্যারেক্টার-পেয়ার একসাথে জোড়া লাগিয়ে লাগিয়ে — ফলে কমন শব্দ (the, is) একটাই টোকেন হয়ে যায়, আর বিরল/নতুন শব্দ ছোট ছোট চেনা টুকরায় ভেঙে যায় (unknown শব্দের সমস্যা প্রায় থাকে না)। GPT, BERT — প্রায় সব আধুনিক LLM-ই BPE বা তার কোনো ভ্যারিয়েন্ট ব্যবহার করে।

নিচের কোডে আমরা HuggingFace-এর `AutoTokenizer` দিয়ে `bert-base-uncased`-এর আসল BPE-স্টাইল (WordPiece, যেটা BPE-র খুব কাছাকাছি একটা ভ্যারিয়েন্ট) টোকেনাইজার লোড করব, এবং ইচ্ছাকৃতভাবে একটা বানানো/বিরল শব্দ (`"supercalifragilisticexpialidocious"`) দিয়ে দেখব কীভাবে সাব-ওয়ার্ডে ভেঙে যায়।

In [ ]:
# pip install transformers  (প্রথমবার চালানোর আগে দরকার হলে uncomment করুন)
# !pip install transformers

from transformers import AutoTokenizer

# BERT-এর টোকেনাইজার লোড করছি (WordPiece — BPE-এর কাছাকাছি একটা সাব-ওয়ার্ড অ্যালগরিদম)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentences = [
    "I love machine learning.",
    "Tokenization splits text into subwords.",
    "supercalifragilisticexpialidocious",   # ইচ্ছাকৃতভাবে বিরল/বানানো শব্দ
]

for sent in sentences:
    tokens = tokenizer.tokenize(sent)
    token_ids = tokenizer.encode(sent)
    print(f"Sentence : {sent}")
    print(f"Tokens   : {tokens}")
    print(f"Token IDs: {token_ids}")
    print("-" * 60)

**খেয়াল করুন:** সাধারণ বাক্যের শব্দগুলো (`love`, `machine`, `learning`) প্রায় হুবহু একটা করে টোকেন হয়ে গেছে, কারণ এগুলো কমন শব্দ — ট্রেনিং কর্পাসে বহুবার দেখা গেছে বলে পুরো শব্দটাই ভোকাবুলারিতে জায়গা পেয়েছে। কিন্তু `supercalifragilisticexpialidocious`-এর মতো বিরল শব্দ ভেঙে গেছে অনেকগুলো ছোট সাব-ওয়ার্ড টুকরায় (`##` প্রিফিক্স মানে "আগের টোকেনের সাথে জোড়া লাগানো, নতুন শব্দ শুরু না")। এভাবেই BPE/WordPiece কখনো সম্পূর্ণ "unknown" শব্দে আটকে যায় না — যেকোনো শব্দকেই চেনা ছোট টুকরায় ভেঙে ফেলতে পারে।

---

### 3.2 Word Embeddings — অর্থকে সংখ্যায় ধরা

একবার টোকেন হয়ে গেলে, প্রতিটা টোকেনকে একটা **fixed-size ভেক্টর** (যেমন ৫০ বা ৩০০ ডাইমেনশনের সংখ্যার লিস্ট) দিয়ে রিপ্রেজেন্ট করা হয় — একে বলে **Embedding**। এই ভেক্টরগুলো এমনভাবে ট্রেইন করা হয় যেন কাছাকাছি অর্থের শব্দ কাছাকাছি ভেক্টরে বসে:

```
vector("king")  - vector("man") + vector("woman")  ≈  vector("queen")
```

* **Word2Vec** — একটা শব্দ তার আশেপাশের শব্দ (context) দেখে প্রেডিক্ট করার চেষ্টা করে ট্রেইন হয়; এই প্রক্রিয়ায় ভেক্টরগুলো নিজে থেকেই অর্থ ধারণ করতে শেখে।
* **GloVe** — পুরো কর্পাসে কোন শব্দ কোন শব্দের কাছাকাছি কতবার এসেছে (co-occurrence statistics) তার ওপর ভিত্তি করে ভেক্টর তৈরি করে।

**সীমাবদ্ধতা:** Word2Vec/GloVe-তে প্রতিটা শব্দের **একটাই** ভেক্টর থাকে, তার মানে "bank" (নদীর তীর) আর "bank" (ব্যাংক) — দুটোই একই ভেক্টর পায়, প্রসঙ্গ (context) যাই হোক না কেন। এই সীমাবদ্ধতা দূর করার জন্যই Transformer-এর জন্ম (নিচে 3.3-এ দেখুন)।

নিচে আমরা `gensim.downloader` দিয়ে ছোট প্রি-ট্রেইন্ড **GloVe** ভেক্টর (`glove-wiki-gigaword-50`, ~৬৫MB) লোড করব। এটা **প্রথমবার রান করার সময়** ডাউনলোড হবে (ইন্টারনেট লাগবে), এরপর থেকে লোকাল ক্যাশ থেকেই লোড হবে — কোনো বারবার ডাউনলোডের দরকার নেই। এটাই "একবার ডাউনলোড, তারপর চিরকাল অফলাইন" — এই কোর্সের "Sovereign AI" দর্শনের মূল কথা।

In [ ]:
# pip install gensim  (প্রথমবার চালানোর আগে দরকার হলে uncomment করুন)
# !pip install gensim

import gensim.downloader as api

# প্রথমবার এই লাইন রান করলে ~65MB ডাউনলোড হবে, পরেরবার থেকে লোকাল ক্যাশ (~/gensim-data) থেকে লোড হবে
glove = api.load("glove-wiki-gigaword-50")

print(f"Vocabulary size: {len(glove.index_to_key)}")
print(f"Vector dimension: {glove.vector_size}")

In [ ]:
# --- Word Analogy: king - man + woman ≈ ? ---
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5)

print("king - man + woman ≈")
for word, score in result:
    print(f"  {word:15s}  similarity={score:.4f}")

In [ ]:
# --- Nearest Neighbors: কোন শব্দগুলো "computer"-এর সবচেয়ে কাছাকাছি? ---
for query_word in ["computer", "happy", "bank"]:
    print(f"\nMost similar to '{query_word}':")
    for word, score in glove.most_similar(query_word, topn=5):
        print(f"  {word:15s}  similarity={score:.4f}")

**খেয়াল করুন — co-occurrence intuition:** GloVe ভেক্টরগুলো তৈরি হয়েছে পুরো কর্পাসে কোন শব্দ কোন শব্দের পাশে কতবার এসেছে তার পরিসংখ্যান থেকে। "computer" শব্দটা যেসব শব্দের পাশে বেশি এসেছে (software, laptop, internet) — সেগুলোই এর কাছাকাছি ভেক্টর পেয়েছে। আর `king - man + woman ≈ queen`-এর মতো এনালজি কাজ করে কারণ "রাজকীয়তা" (royalty) আর "লিঙ্গ" (gender) — এই দুটো ধারণা ভেক্টর স্পেসে প্রায় সরলরৈখিক দিকে (linear direction) এনকোড হয়ে গেছে, নিছক কাকতালীয়ভাবে না, বরং কারণ কর্পাসে "king" আর "queen" একই ধরনের প্রসঙ্গে ব্যবহার হয়, শুধু লিঙ্গ-নির্দেশক শব্দগুলো ভিন্ন।

তবে `bank`-এর নিকটতম প্রতিবেশীদের দিকে তাকান — GloVe শুধু **একটা** ভেক্টর দিতে পারে, তাই "নদীর তীর" আর "ব্যাংক" দুই অর্থই একসাথে মিশে যায়। এই সীমাবদ্ধতাই পরের সেকশনের মূল বিষয়।

---

### 3.3 Transformer Attention — প্রসঙ্গ বোঝা

Transformer একটা বাক্যের প্রতিটা শব্দের ভেক্টরকে **বাক্যের বাকি শব্দগুলোর প্রেক্ষাপটে** নতুন করে হিসাব করে — একে বলে **Self-Attention**। "bank" শব্দটা "river"-এর কাছে থাকলে একরকম ভেক্টর পাবে, "money"-এর কাছে থাকলে সম্পূর্ণ ভিন্ন ভেক্টর পাবে।

```
Query (আমি কী খুঁজছি) · Key (প্রতিটা শব্দ কী অফার করছে) ──▶ Attention Score
Attention Score → Softmax → Weighted sum of Value vectors
```

এই মেকানিজমের মূল অঙ্কটা আসলে Week 2-এর **Scaled Dot-Product** — $\dfrac{Q \cdot K^T}{\sqrt{d_k}}$। প্রতিটা শব্দ (Query) বাক্যের বাকি সব শব্দের (Key) সাথে ডট প্রোডাক্ট করে দেখে কার সাথে কতটা "সম্পর্কিত"। যে শব্দগুলোর সাথে সম্পর্ক বেশি (dot product বেশি), তাদের ভ্যালু (Value) বেশি ওজন পেয়ে ফাইনাল রিপ্রেজেন্টেশনে যোগ হয়। এআই যেভাবে "চিন্তা করে" — অর্থাৎ কোন শব্দ কোন শব্দের সাথে সংযুক্ত করে অর্থ বুঝছে — তার মূলে এই ডট প্রোডাক্ট।

এই নোটবুকে আমরা পুরো Transformer আর্কিটেকচার ট্রেইন করছি না (সেটা নিজেই একটা আলাদা লেকচারের বিষয়), কিন্তু নিচের সেকশনে (3.4) আমরা দেখব একটা **প্রি-ট্রেইন্ড Transformer-based sentence encoder** ব্যবহার করলে GloVe-এর "একটা শব্দ = একটা ভেক্টর" সীমাবদ্ধতা কীভাবে কেটে যায়, কারণ পুরো বাক্যটাকেই একসাথে এনকোড করা হয়, প্রতিটা শব্দ বাকি বাক্যের প্রসঙ্গ নিয়েই।

---

### 3.4 Sentence Embeddings — পুরো বাক্যকে একটা ভেক্টরে ধরা

আমাদের প্রজেক্টের জন্য দরকার একটা শব্দ না, বরং পুরো CV বা জব ডেসক্রিপশনের একটা ভেক্টর। **Sentence-Transformers** মডেলগুলো (BERT-এর ওপর ভিত্তি করে তৈরি, Self-Attention ব্যবহার করেই) একটা পুরো প্যারাগ্রাফ নিয়ে একটামাত্র ফিক্সড-সাইজ ভেক্টর আউটপুট দেয়, যেটাতে পুরো টেক্সটের অর্থ সংকুচিত হয়ে থাকে। দুটো টেক্সটের ভেক্টরের মধ্যে **Cosine Similarity** বের করলেই বোঝা যায় তারা অর্থগতভাবে কতটা কাছাকাছি।

**Week 2-এর সাথে সংযোগ:** মনে করুন Week 2-তে আমরা NumPy দিয়ে শিখেছিলাম দুটো ভেক্টরের **Dot Product** এবং **Cosine Similarity** কীভাবে বের করতে হয় — $\cos(\theta) = \dfrac{A \cdot B}{\|A\| \|B\|}$। এখানে ঠিক সেই একই সূত্র ব্যবহার হচ্ছে, শুধু ভেক্টরগুলো এখন র‍্যান্ডম সংখ্যা না, বরং একটা পুরো CV বা জব ডেসক্রিপশনের **অর্থ** ধারণ করছে। Cosine Similarity ব্যবহার করা হয় কারণ এটা ভেক্টরের **ম্যাগনিচিউড** (length, অর্থাৎ টেক্সটের length) নয়, শুধু **দিক** (direction, অর্থাৎ অর্থ) মাপে — একটা লম্বা CV আর একটা ছোট জব ডেসক্রিপশন ভিন্ন length-এর হলেও, একই বিষয় নিয়ে কথা বললে তাদের ভেক্টরের দিক কাছাকাছি থাকবে।

নিচের কোডে আমরা `paraphrase-MiniLM-L3-v2` মডেল দিয়ে একটা জব ডেসক্রিপশন আর তিনটা CV স্নিপেট এমবেড করব, তারপর **ম্যানুয়ালি NumPy দিয়ে** dot-product/norm ফর্মুলা লিখে cosine similarity বের করব (Week 2-এর রিমাইন্ডার হিসেবে), এবং শেষে দেখাব `sentence_transformers.util.cos_sim` দিয়ে একই কাজ এক লাইনে কীভাবে করা যায়।

---

## 4. Details & Valid Points

### 4.1 Model Reference Table — কোন এমবেডিং মডেল কখন ব্যবহার করবেন

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| **`paraphrase-MiniLM-L3-v2`** (এই নোটবুকে ব্যবহৃত) | দ্রুত, লোকাল সেমান্টিক সিমিলারিটি | ছোট (~৬১MB), CPU-তেই ফাস্ট | বড় মডেলের তুলনায় সামান্য কম নির্ভুল |
| `all-MiniLM-L6-v2` | জেনারেল-পারপাস সেন্টেন্স এমবেডিং | ভালো অ্যাকুরেসি, তবু ছোট | L3 ভ্যারিয়েন্টের চেয়ে ~৫০% বড়, একটু ধীর |
| `all-mpnet-base-v2` | হাই-অ্যাকুরেসি সেমান্টিক সার্চ | এই ফ্যামিলিতে সবচেয়ে নির্ভুল | অনেক ভারী (~৪২০MB), CPU-তে ধীর |
| OpenAI `text-embedding-3-small` | ক্লাউড-বেসড এমবেডিং | লোকাল কম্পিউট লাগে না | লোকাল না — প্রতি কলে খরচ, ডেটা থার্ড-পার্টিতে যায় |

> **Valid Point:** ছোট মডেল মানেই "খারাপ" না — একটা পোর্টফোলিও-স্কেল CV ম্যাচারে `paraphrase-MiniLM-L3-v2`-এর অ্যাকুরেসি যথেষ্ট, আর CPU-তেই রিয়েল-টাইম রেসপন্স পাওয়া যায়। প্রোডাকশনে ইউজার বেশি এবং অ্যাকুরেসি critical হলে `all-mpnet-base-v2`-এর দিকে যাওয়া যুক্তিসঙ্গত।

`paraphrase-MiniLM-L3-v2` HuggingFace Hub থেকে **প্রথমবার রান করার সময়** ডাউনলোড হয় (ইন্টারনেট লাগবে), কিন্তু এরপর থেকে এটা সম্পূর্ণ **লোকাল**ভাবে চলে — কোনো ইন্টারনেট বা এপিআই কল লাগে না। "একবার ডাউনলোড, তারপর চিরকাল অফলাইন" — ডেটা নিজের মেশিনে থাকে, কোনো থার্ড-পার্টি সার্ভারে যায় না।

---

## 5. Achievement: Sentence Embedding + Cosine Similarity Mini-Demo

এখন আমরা হাতে-কলমে দেখব সেমান্টিক ম্যাচিং কীভাবে কাজ করে — একটা টয় জব ডেসক্রিপশন এবং তিনটা টয় CV স্নিপেট (একটা স্পষ্টভাবে প্রাসঙ্গিক, একটা কিছুটা প্রাসঙ্গিক, একটা সম্পূর্ণ অপ্রাসঙ্গিক) নিয়ে।

In [ ]:
# pip install sentence-transformers  (প্রথমবার চালানোর আগে দরকার হলে uncomment করুন)
# !pip install sentence-transformers

import numpy as np
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

job_description = (
    "We are looking for a Machine Learning Engineer with experience in Python, "
    "deep learning, and NLP to build production-grade AI systems."
)

cv_snippets = {
    "Relevant CV": (
        "Experienced ML engineer skilled in Python, PyTorch, and NLP. "
        "Built and deployed several deep learning models to production."
    ),
    "Loosely Relevant CV": (
        "Software developer with 3 years of experience building web applications "
        "using JavaScript and some exposure to data analysis."
    ),
    "Irrelevant CV": (
        "Professional chef with 10 years of experience in French cuisine, "
        "restaurant management, and menu design."
    ),
}

print("Job Description:")
print(f"  {job_description}\n")
for name, text in cv_snippets.items():
    print(f"{name}: {text}")

In [ ]:
# --- Embed everything ---
job_vec = model.encode(job_description)
cv_vecs = {name: model.encode(text) for name, text in cv_snippets.items()}

print(f"Embedding dimension: {job_vec.shape[0]}")

In [ ]:
# --- Manual cosine similarity: Week 2-এর dot-product/norm ফর্মুলা মনে করিয়ে দেওয়ার জন্য ---
def cosine_similarity_manual(a, b):
    dot_product = np.dot(a, b)          # A · B
    norm_a = np.linalg.norm(a)          # ||A||
    norm_b = np.linalg.norm(b)          # ||B||
    return dot_product / (norm_a * norm_b)

print("Cosine Similarity (manual NumPy dot-product/norm):\n")
scores_manual = {}
for name, vec in cv_vecs.items():
    score = cosine_similarity_manual(job_vec, vec)
    scores_manual[name] = score
    print(f"  {name:22s}: {score:.4f}")

# --- Rank CVs by similarity, most relevant first ---
ranked = sorted(scores_manual.items(), key=lambda kv: kv[1], reverse=True)
print("\nRanking (best match first):")
for rank, (name, score) in enumerate(ranked, start=1):
    print(f"  {rank}. {name:22s} (score={score:.4f})")

In [ ]:
# --- একই কাজ built-in utility দিয়ে (sentence-transformers.util.cos_sim) ---
job_tensor = model.encode(job_description, convert_to_tensor=True)

print("Cosine Similarity (sentence_transformers.util.cos_sim):\n")
for name, text in cv_snippets.items():
    cv_tensor = model.encode(text, convert_to_tensor=True)
    score = util.cos_sim(job_tensor, cv_tensor).item()
    print(f"  {name:22s}: {score:.4f}")

**খেয়াল করুন:** ম্যানুয়াল NumPy ক্যালকুলেশন আর বিল্ট-ইন `util.cos_sim`-এর স্কোর প্রায় হুবহু মিলে যাচ্ছে — কারণ ভেতরে দুটোই একই গণিত করছে ($\cos(\theta) = \frac{A \cdot B}{\|A\|\|B\|}$), শুধু `util.cos_sim` ব্যাচ-প্রসেসিং আর GPU টেন্সরের জন্য অপ্টিমাইজড। "Relevant CV"-এর স্কোর সবচেয়ে বেশি হওয়া উচিত, "Irrelevant CV"-এর সবচেয়ে কম — এটাই প্রমাণ করে যে এমবেডিং মডেলটা সত্যিই **অর্থ** ধরতে পারছে, শুধু কমন শব্দ (যেমন "experience") গুনে না।

এই একই প্যাটার্ন — এমবেড করো, cosine similarity বের করো, র‍্যাঙ্ক করো — স্কেল করলেই হয়ে যায় `Class 1 Project/`-এর সম্পূর্ণ Semantic CV-to-Job Matcher অ্যাপ (একাধিক CV, PDF/DOCX আপলোড সহ)।

---

## 6. Summary

আজকে আমরা NLP পাইপলাইনের তিনটা মূল ধাপ হাতে-কলমে দেখলাম:

1. **Tokenization (BPE/WordPiece)** — টেক্সটকে সাব-ওয়ার্ড টোকেনে ভাঙা, যাতে কমন শব্দ কমপ্যাক্ট থাকে আর বিরল শব্দও চেনা টুকরায় ভেঙে ফেলা যায়।
2. **Word Embeddings (GloVe)** — প্রতিটা শব্দকে একটা ভেক্টরে রূপান্তর করা, যেখানে co-occurrence statistics-এর কারণে কাছাকাছি অর্থের শব্দ কাছাকাছি ভেক্টরে বসে; word-analogy অঙ্ক (`king - man + woman ≈ queen`) দেখলাম।
3. **Sentence Embeddings + Cosine Similarity** — একটা পুরো বাক্য/প্যারাগ্রাফকে ভেক্টরে রূপান্তর করে (Transformer Attention ব্যবহার করে, যা GloVe-এর "এক শব্দ = এক ভেক্টর" সীমাবদ্ধতা কাটিয়ে ওঠে) Week 2-এর dot-product/cosine-similarity ফর্মুলা দিয়ে অর্থগত মিল মাপা — এটাই সেমান্টিক CV ম্যাচিং-এর মূল ভিত্তি।

---

## 🧠 Brain Teasers & Exercises (নিজে চেষ্টা করুন)

1. **BPE Merge Order:** BPE ট্রেইন করার সময় সবচেয়ে ফ্রিকোয়েন্ট ক্যারেক্টার-পেয়ার আগে মার্জ হয়। উপরের কোডে `tokenizer.tokenize()`-এ আরও ২-৩টা বিরল/টেকনিক্যাল শব্দ (যেমন `"antidisestablishmentarianism"`) পাস করে দেখুন কীভাবে ভাঙে, এবং টুকরাগুলো কেন ঠিক ওখানেই ভাঙল তা অনুমান করার চেষ্টা করুন।
2. **Word2Vec/GloVe Analogy Failure:** `king - man + woman ≈ queen`-এর মতো এনালজি সবসময় কাজ করে না। `glove.most_similar()` ব্যবহার করে এমন একটা এনালজি টেস্ট করুন যেটা ফেল করতে পারে (যেমন সাংস্কৃতিকভাবে নির্দিষ্ট কোনো সম্পর্ক, বা বাংলাদেশ-প্রসঙ্গের কোনো শব্দ-জোড়া যা ইংরেজি Wikipedia কর্পাসে কম দেখা গেছে), এবং ফলাফল দেখে ব্যাখ্যা করুন কেন ফেল করল।
3. **Attention vs. GloVe:** "The bank raised interest rates" আর "We sat by the river bank" — এই দুই বাক্যে `glove["bank"]` একটাই ভেক্টর দেবে, কিন্তু Sentence-Transformers পুরো বাক্য এনকোড করলে আলাদা রেজাল্ট দেবে। উপরের `model.encode()` ব্যবহার করে দুই বাক্যকে সরাসরি এমবেড করে তাদের মধ্যে cosine similarity বের করুন, তারপর "river" আর "money" শব্দ দুটো বাক্যের সাথে আলাদাভাবে মিলিয়ে দেখুন কোনটার সাথে বেশি মিলছে — এটাই দেখাবে Attention কীভাবে context ধরে।